In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
from datetime import datetime, date
DatetimeIndex = pd.DatetimeIndex
from dateutil.relativedelta import relativedelta
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- dates_pol_interval ---
def FIX_DATES_POL_INTERVAL_ARG_MATCH(name, value, choices):
    if value not in choices:
        raise ValueError(f"{name} must be one of {choices}")

def FIX_DATES_POL_INTERVAL_LEN2(value):
    if isinstance(value, (pd.Series, pl.Series, pd.DatetimeIndex, list, tuple, np.ndarray)):
        return len(value)
    return 1

def _convert_date(value):
    if isinstance(value, pl.Series):
        value = value.to_list()
    if isinstance(value, (pd.Series, pd.DatetimeIndex, list, tuple, np.ndarray)):
        converted = pd.to_datetime(value)
        if hasattr(converted, "to_pydatetime"):
            return list(converted.to_pydatetime())
        return list(converted.dt.to_pydatetime())
    converted = pd.to_datetime(value)
    return converted.to_pydatetime() if hasattr(converted, "to_pydatetime") else converted

FIX_DATES_POL_INTERVAL_MTH_CALC = None
FIX_DATES_POL_INTERVAL_RELATIVEDELTA = relativedelta

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_dates_pol_interval(arg_match, len2, mth_calc, relativedelta):
    def pol_interval(dates: str | datetime | DatetimeIndex | pd.Series,
                     issue_date: str | datetime | DatetimeIndex | pd.Series,
                     dur_length: str) -> np.ndarray:
        arg_match('dur_length', dur_length, ['year', 'quarter', 'month', 'week'])

        dates = _convert_date(dates)
        issue_date = _convert_date(issue_date)

        dat = pd.DataFrame({
            'issue_date': issue_date,
            'dates': dates
        }, index=np.arange(max(len2(dates), len2(issue_date))))

        if dur_length == "year":
            res = [relativedelta(a, b).years for a, b in
                   zip(dat.dates, dat.issue_date)]

        elif dur_length in ["month", "quarter"]:
            def mth_calc(a, b):
                delta = relativedelta(a, b)
                return 12 * delta.years + delta.months

            if dur_length == "quarter":
                res = [mth_calc(a, b) // 3 for a, b
                       in zip(dat.dates, dat.issue_date)]
            else:
                res = [mth_calc(a, b) for a, b in zip(dat.dates, dat.issue_date)]

        else:
            res = (dat.dates - dat.issue_date).dt.days // 7

        return np.array(res) + 1
    return pol_interval

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_dates_pol_interval(arg_match, len2, mth_calc, relativedelta):
    import numpy as np
    from pandas.core.indexes.datetimes import DatetimeIndex

    def pol_interval(dates: str | datetime | DatetimeIndex | pd.Series,
                     issue_date: str | datetime | DatetimeIndex | pd.Series,
                     dur_length: str) -> np.ndarray:
        arg_match('dur_length', dur_length, ['year', 'quarter', 'month', 'week'])

        dates = _convert_date(dates)
        issue_date = _convert_date(issue_date)

        n = max(len2(dates), len2(issue_date))

        def _to_list(x, n):
            if isinstance(x, pl.Series):
                vals = x.to_list()
            elif isinstance(x, (pd.Series, DatetimeIndex, np.ndarray, list, tuple)):
                vals = list(x)
            else:
                return [x] * n
            if len(vals) < n:
                vals = vals + [None] * (n - len(vals))
            return vals

        dates = _to_list(dates, n)
        issue_date = _to_list(issue_date, n)

        dat = pl.DataFrame({
            'issue_date': issue_date,
            'dates': dates
        })

        if dur_length == "year":
            res = [relativedelta(a, b).years for a, b in
                   zip(dat["dates"].to_list(), dat["issue_date"].to_list())]

        elif dur_length in ["month", "quarter"]:
            def mth_calc(a, b):
                delta = relativedelta(a, b)
                return 12 * delta.years + delta.months

            if dur_length == "quarter":
                res = [mth_calc(a, b) // 3 for a, b
                       in zip(dat["dates"].to_list(), dat["issue_date"].to_list())]
            else:
                res = [mth_calc(a, b) for a, b in zip(dat["dates"].to_list(), dat["issue_date"].to_list())]

        else:
            res = (pl.Series(dat["dates"]) - pl.Series(dat["issue_date"])).dt.total_days().to_list()
            res = [x // 7 if x is not None else None for x in res]

        return np.array(res) + 1
    return pol_interval

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: dates_pol_interval ===

# L1 smoke – generated
try:
    _r = gen_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    print("✅ L1 smoke gen_dates_pol_interval: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_dates_pol_interval: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    print("✅ L1 smoke before_dates_pol_interval: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_dates_pol_interval: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – call the returned function on the main date-vector path.
try:
    _bf = before_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _gf = gen_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _dates = pd.Series(pd.to_datetime(["2021-01-01", "2021-07-01", "2022-01-15"]))
    _issues = pd.Series(pd.to_datetime(["2020-01-01", "2021-01-01", "2021-12-15"]))
    _before_main = _bf(_dates, _issues, "month")
    _gen_main = _gf(pl.Series(_dates), pl.Series(_issues), "month")
    if np.array_equal(_before_main, _gen_main):
        print("✅ L2 equivalence dates_pol_interval month vector: MATCH")
    else:
        print(f"❌ L2 equivalence dates_pol_interval month vector: MISMATCH — before={_before_main}, gen={_gen_main}")
except Exception as _e:
    print(f"❌ L2 equivalence dates_pol_interval: setup error — {type(_e).__name__}: {_e}")

# L3 branch – execute returned function across year/month/quarter/week and invalid input.
try:
    _bf = before_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _gf = gen_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _dates = pd.Series(pd.to_datetime(["2021-01-01", "2021-07-01", "2022-01-15"]))
    _issues = pd.Series(pd.to_datetime(["2020-01-01", "2021-01-01", "2021-12-15"]))
    for _dur in ["year", "month", "quarter", "week"]:
        _before_edge = _bf(_dates, _issues, _dur)
        _gen_edge = _gf(pl.Series(_dates), pl.Series(_issues), _dur)
        if np.array_equal(_before_edge, _gen_edge):
            print(f"✅ L3 branch dates_pol_interval {_dur}: MATCH")
        else:
            print(f"❌ L3 branch dates_pol_interval {_dur}: MISMATCH — before={_before_edge}, gen={_gen_edge}")
    try:
        _bf(_dates, _issues, "day")
        _before_exc = None
    except Exception as _e:
        _before_exc = type(_e).__name__
    try:
        _gf(pl.Series(_dates), pl.Series(_issues), "day")
        _gen_exc = None
    except Exception as _e:
        _gen_exc = type(_e).__name__
    if _before_exc == _gen_exc:
        print(f"✅ L3 invalid dates_pol_interval: MATCH ({_before_exc})")
    else:
        print(f"❌ L3 invalid dates_pol_interval: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 branch dates_pol_interval: {type(_e).__name__}: {_e}")

# L3 edge – scalar string inputs should behave the same as one-row vector inputs.
try:
    _bf = before_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _gf = gen_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _before_scalar = _bf("2021-02-15", "2020-02-15", "year")
    _gen_scalar = _gf("2021-02-15", "2020-02-15", "year")
    if np.array_equal(_before_scalar, _gen_scalar):
        print(f"✅ L3 edge dates_pol_interval scalar strings: MATCH {_gen_scalar}")
    else:
        print(f"❌ L3 edge dates_pol_interval scalar strings: MISMATCH — before={_before_scalar}, gen={_gen_scalar}")
except Exception as _e:
    print(f"❌ L3 edge dates_pol_interval scalar strings: {type(_e).__name__}: {_e}")

# L3 edge – mismatched lengths should expose whether scalar/vector expansion matches pandas behavior.
try:
    _bf = before_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _gf = gen_dates_pol_interval(FIX_DATES_POL_INTERVAL_ARG_MATCH, FIX_DATES_POL_INTERVAL_LEN2, FIX_DATES_POL_INTERVAL_MTH_CALC, FIX_DATES_POL_INTERVAL_RELATIVEDELTA)
    _dates = pd.Series(pd.to_datetime(["2021-01-01", "2021-02-01"]))
    _issue = "2020-01-01"
    _before_exc = _gen_exc = None
    try:
        _before_mismatch = _bf(_dates, _issue, "month")
    except Exception as _e:
        _before_exc = type(_e).__name__
        _before_mismatch = None
    try:
        _gen_mismatch = _gf(pl.Series(_dates), _issue, "month")
    except Exception as _e:
        _gen_exc = type(_e).__name__
        _gen_mismatch = None
    if _before_exc or _gen_exc:
        if _before_exc == _gen_exc:
            print(f"✅ L3 edge dates_pol_interval mismatched lengths: MATCH exception ({_before_exc})")
        else:
            print(f"❌ L3 edge dates_pol_interval mismatched lengths: MISMATCH exception — before={_before_exc}, gen={_gen_exc}")
    elif np.array_equal(_before_mismatch, _gen_mismatch):
        print(f"✅ L3 edge dates_pol_interval mismatched lengths: MATCH {_gen_mismatch}")
    else:
        print(f"❌ L3 edge dates_pol_interval mismatched lengths: MISMATCH — before={_before_mismatch}, gen={_gen_mismatch}")
except Exception as _e:
    print(f"❌ L3 edge dates_pol_interval mismatched lengths: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_dates_pol_interval: OK, type= function
✅ L1 smoke before_dates_pol_interval: OK
✅ L2 equivalence dates_pol_interval month vector: MATCH
✅ L3 branch dates_pol_interval year: MATCH
✅ L3 branch dates_pol_interval month: MATCH
✅ L3 branch dates_pol_interval quarter: MATCH
✅ L3 branch dates_pol_interval week: MATCH
✅ L3 invalid dates_pol_interval: MATCH (ValueError)
✅ L3 edge dates_pol_interval scalar strings: MATCH [2]
✅ L3 edge dates_pol_interval mismatched lengths: MATCH [13 14]
